#### Session 3: Multi-model forecasting group

This notebook compares a diverse set of forecasting models, from simple baselines to classical statistical methods and Prophet.

The goal is not only to measure accuracy, but also to create meaningful variation in error patterns for interpretation, bias analysis, and explainability.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
from pathlib import Path

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX

try:
    from prophet import Prophet
    HAS_PROPHET = True
except ImportError:
    HAS_PROPHET = False
    print("Prophet not installed — skipping Prophet model")

from sklearn.linear_model import LinearRegression


## Load data

The series is loaded as a univariate time series with a `DatetimeIndex`.  
For consistency across all models, the data is resampled to daily frequency.

In [2]:
DATA_DIR = Path("../src/data")
OUTPUT_DIR = Path("../src/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "energy"


In [3]:
def load_series() -> pd.Series:
    csv_path = DATA_DIR / "raw" / "AEP_hourly.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path, parse_dates=["Datetime"], index_col="Datetime")
        s = df.squeeze().sort_index()
    else:
        rng = np.random.default_rng(42)
        idx = pd.date_range("2020-01-01", periods=8760, freq="h")
        trend = np.linspace(12000, 14000, 8760)
        seasonal_daily = 2000 * np.sin(2 * np.pi * np.arange(8760) / 24)
        seasonal_weekly = 800 * np.sin(2 * np.pi * np.arange(8760) / (24 * 7))
        noise = rng.normal(0, 300, 8760)
        s = pd.Series(trend + seasonal_daily + seasonal_weekly + noise, index=idx, name="consumption_MW")

    s = s.resample("D").mean().dropna()
    return s

series = load_series()
print(f"Series length: {len(series)} | freq: {pd.infer_freq(series.index)}")


Series length: 365 | freq: D


## Train-test split

The last 30 daily observations are reserved for testing.  
This keeps the evaluation simple and makes the forecast horizon easy to interpret.


In [4]:
def compute_metrics(actual, predicted, model_name):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    safe_actual = np.where(actual == 0, np.nan, actual)

    mae = float(np.nanmean(np.abs(actual - predicted)))
    rmse = float(np.sqrt(np.nanmean((actual - predicted) ** 2)))
    mape = float(np.nanmean(np.abs((actual - predicted) / safe_actual)) * 100)
    mpe = float(np.nanmean((actual - predicted) / safe_actual) * 100)

    if len(actual) > 1:
        dir_actual = np.sign(np.diff(actual))
        dir_predicted = np.sign(np.diff(predicted))
        da = float(np.mean(dir_actual == dir_predicted) * 100)
    else:
        da = float("nan")

    return {
        "model": model_name,
        "MAE": round(mae, 2),
        "RMSE": round(rmse, 2),
        "MAPE": round(mape, 4),
        "MPE": round(mpe, 4),
        "DA": round(da, 2),
    }

def store_forecast(name, forecast):
    records = []
    for i, (idx, act) in enumerate(test.items()):
        records.append({
            "date": str(idx.date()),
            "actual": round(float(act), 2),
            "predicted": round(float(forecast[i]), 2),
            "model": name,
        })
    return records

all_metrics = []
all_forecasts = []


In [5]:
TEST_PERIODS = 30

series = load_series()
train = series.iloc[:-TEST_PERIODS]
test = series.iloc[-TEST_PERIODS:]
horizon = len(test)

all_metrics = []
all_forecasts = []

print(f"Series length: {len(series)} | Train: {len(train)} | Test: {len(test)}")

Series length: 365 | Train: 335 | Test: 30


## Model 1: Naive baseline

This forecast repeats the last observed value and serves as the simplest benchmark.


In [6]:
last_value = float(train.iloc[-1])
naive_pred = np.full(horizon, last_value)

all_metrics.append(compute_metrics(test.values, naive_pred, "Naive"))
all_forecasts.extend(store_forecast("Naive", naive_pred))
print("✓ Naive")


✓ Naive


## Model 2: Seasonal Naive baseline

This model reuses the value from the same point in the previous weekly cycle and captures recurring seasonality.


In [7]:
SEASON = 7

snaive_pred = np.array([
    train.iloc[-SEASON + (i % SEASON)]
    for i in range(horizon)
])

all_metrics.append(compute_metrics(test.values, snaive_pred, "Seasonal Naive"))
all_forecasts.extend(store_forecast("Seasonal Naive", snaive_pred))
print("✓ Seasonal Naive")


✓ Seasonal Naive


## Model 3: Linear Regression trend baseline

This model captures only a linear trend, without explicit seasonality.


In [8]:
X_train = np.arange(len(train)).reshape(-1, 1)
X_test = np.arange(len(train), len(train) + horizon).reshape(-1, 1)

lr = LinearRegression().fit(X_train, train.values)
lr_pred = lr.predict(X_test)

all_metrics.append(compute_metrics(test.values, lr_pred, "Linear Regression"))
all_forecasts.extend(store_forecast("Linear Regression", lr_pred))
print("✓ Linear Regression")


✓ Linear Regression


## Model 4: ETS

ETS combines error, trend, and seasonality through exponential smoothing.


In [9]:
ets_model = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=SEASON,
    initialization_method="estimated",
)
ets_fit = ets_model.fit(optimized=True)
ets_pred = ets_fit.forecast(horizon).values

all_metrics.append(compute_metrics(test.values, ets_pred, "ETS"))
all_forecasts.extend(store_forecast("ETS", ets_pred))
print("✓ ETS")


✓ ETS


## Model 5: Holt-Winters Exponential Smoothing

This version uses a damped trend so the forecast growth tapers off over time.


In [10]:
hwes_model = ExponentialSmoothing(
    train,
    trend="add",
    damped_trend=True,
    seasonal="add",
    seasonal_periods=SEASON,
    initialization_method="estimated",
)
hwes_fit = hwes_model.fit(optimized=True)
hwes_pred = hwes_fit.forecast(horizon).values

all_metrics.append(compute_metrics(test.values, hwes_pred, "HWES (damped)"))
all_forecasts.extend(store_forecast("HWES (damped)", hwes_pred))
print("✓ HWES (damped)")


✓ HWES (damped)


## Model 6: SARIMA

SARIMA extends ARIMA with seasonal structure and is often useful when repeated cycles matter.


In [11]:
try:
    sarima_train = train.iloc[-min(730, len(train)):]
    sarima_model = SARIMAX(
        sarima_train,
        order=(1, 1, 1),
        seasonal_order=(1, 1, 1, SEASON),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    sarima_fit = sarima_model.fit(disp=False)
    sarima_pred = sarima_fit.forecast(steps=horizon).values

    all_metrics.append(compute_metrics(test.values, sarima_pred, "SARIMA"))
    all_forecasts.extend(store_forecast("SARIMA", sarima_pred))
    print("✓ SARIMA")
except Exception as e:
    print(f"✗ SARIMA failed: {e}")


✓ SARIMA


## Model 7: Prophet

Prophet is included as a flexible decomposable model that can capture trend and seasonality patterns in a different way from the classical methods.


In [12]:
if HAS_PROPHET:
    try:
        prophet_df = pd.DataFrame({"ds": train.index, "y": train.values})
        prophet_model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True,
            daily_seasonality=False,
            interval_width=0.95,
        )
        prophet_model.fit(prophet_df)

        future = prophet_model.make_future_dataframe(periods=horizon, freq="D")
        forecast = prophet_model.predict(future)
        prophet_pred = forecast["yhat"].iloc[-horizon:].values

        all_metrics.append(compute_metrics(test.values, prophet_pred, "Prophet"))
        all_forecasts.extend(store_forecast("Prophet", prophet_pred))
        print("✓ Prophet")
    except Exception as e:
        print(f"✗ Prophet failed: {e}")
else:
    print("⚠ Prophet skipped (not installed)")


15:37:54 - cmdstanpy - INFO - Chain [1] start processing
15:37:54 - cmdstanpy - INFO - Chain [1] done processing


✓ Prophet


## Save outputs

The metrics and forecasts are saved for later interpretation and for any downstream visualization layer.


In [13]:
metrics_df = pd.DataFrame(all_metrics)

metrics_path = OUTPUT_DIR / "metrics_all_models.json"
metrics_df.to_json(metrics_path, orient="records", indent=2)

forecasts_path = OUTPUT_DIR / "forecasts_all_models.json"
with open(forecasts_path, "w") as f:
    json.dump(all_forecasts, f, indent=2)

metrics_df.to_csv(OUTPUT_DIR / "metrics_all_models.csv", index=False)

print(metrics_df.to_string(index=False))
print(f"Saved metrics → {metrics_path}")
print(f"Saved forecasts → {forecasts_path}")


            model    MAE   RMSE   MAPE     MPE    DA
            Naive 781.31 948.15 5.4682  5.4343  0.00
   Seasonal Naive 130.40 152.60 0.9289  0.8920 96.55
Linear Regression 483.29 547.70 3.4717 -0.0150 48.28
              ETS  44.67  57.32 0.3199  0.0488 96.55
    HWES (damped)  62.69  79.08 0.4478  0.4044 96.55
           SARIMA  45.66  59.78 0.3258  0.1423 96.55
          Prophet  89.01 112.18 0.6366  0.6106 96.55
Saved metrics → ../src/data/metrics_all_models.json
Saved forecasts → ../src/data/forecasts_all_models.json


## Ranking and bias interpretation

The table below ranks models by MAE and summarizes forecast bias using MPE.


In [14]:
print("\n=== Ranking by MAE (lower is better) ===")
ranked = metrics_df.sort_values("MAE").reset_index(drop=True)
ranked.index += 1
print(ranked[["model", "MAE", "RMSE", "MAPE", "MPE", "DA"]].to_string())

print("\n=== Bias check (MPE: + = over-forecast, − = under-forecast) ===")
for _, row in metrics_df.iterrows():
    bias_label = "over-forecast" if row["MPE"] > 1 else ("under-forecast" if row["MPE"] < -1 else "unbiased")
    print(f"  {row['model']:25s}  MPE={row['MPE']:+.2f}%  → {bias_label}")



=== Ranking by MAE (lower is better) ===
               model     MAE    RMSE    MAPE     MPE     DA
1                ETS   44.67   57.32  0.3199  0.0488  96.55
2             SARIMA   45.66   59.78  0.3258  0.1423  96.55
3      HWES (damped)   62.69   79.08  0.4478  0.4044  96.55
4            Prophet   89.01  112.18  0.6366  0.6106  96.55
5     Seasonal Naive  130.40  152.60  0.9289  0.8920  96.55
6  Linear Regression  483.29  547.70  3.4717 -0.0150  48.28
7              Naive  781.31  948.15  5.4682  5.4343   0.00

=== Bias check (MPE: + = over-forecast, − = under-forecast) ===
  Naive                      MPE=+5.43%  → over-forecast
  Seasonal Naive             MPE=+0.89%  → unbiased
  Linear Regression          MPE=-0.01%  → unbiased
  ETS                        MPE=+0.05%  → unbiased
  HWES (damped)              MPE=+0.40%  → unbiased
  SARIMA                     MPE=+0.14%  → unbiased
  Prophet                    MPE=+0.61%  → unbiased


## Interpretation

The main purpose of this notebook is to compare model behavior across a diverse forecasting group.

The next step is to interpret not only which model performs best, but also which models are biased, too smooth, or too reactive.


#### - Naive is the simplest reference point and usually performs worst unless the series is very stable.
- Seasonal Naive often improves on Naive when weekly repetition is strong.
- Linear Regression captures trend but misses local seasonality.
- ETS and HWES are useful for smooth trend-seasonality structure, with HWES usually being more conservative because of damped trend.
- SARIMA can model richer autocorrelation and seasonality, but may be slower and more sensitive to parameter choice.
- Prophet is useful when the series has structured trend and seasonal components and often provides a strong comparison point for interpretation.
